# Homework - Grover MaxCut

The places where you have enter code are marked with `# YOUR CODE HERE`.

In [1]:
!pip install cirq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 51.9 MB/s eta 0:00:00


In [2]:
import cirq
from cirq import H, X, Y, Z, CX, CCX, inverse

## Question 1 (4 points)

Write a function, `oracle010`, that implements an oracle that marks the state $|010 \rangle$. The function `oracle010` has

* input: `qq`, a 3-qubit register
* returns: `None`

The function should append a sequence of gates to `qq` to mark the state $|010\rangle$ only. Don't append any measurements to `qq`.

To help you test the function, we have provided the `grover_diffusion` and `grover` functions.

In [3]:
def oracle010(qq):
    # Flip the qubits that should be |0> to make them |1>
    circuit = cirq.Circuit()

    circuit.append(cirq.X(qq[0]))   # q0 should be 0 → flip to 1
    circuit.append(cirq.X(qq[2]))   # q2 should be 0 → flip to 1

    # Apply CCZ (phase flip when all three qubits are |1>)
    circuit.append(cirq.H(qq[1]))
    circuit.append(cirq.CCX(qq[0], qq[2], qq[1]))   # Toffoli
    circuit.append(cirq.H(qq[1]))

    # Uncompute the X gates
    circuit.append(cirq.X(qq[2]))
    circuit.append(cirq.X(qq[0]))

    return circuit

In [4]:
# visualize your implemented gates
qqTest = cirq.LineQubit.range(3)
circuit = cirq.Circuit()
circuit.append(oracle010(qqTest))
circuit

0: ───X───@───X───
          │
1: ───H───X───H───
          │
2: ───X───@───X───

In [5]:
# To check your solution, we need some to implement grover
def grover_diffusion(qq,n):
    yield H.on_each(*qq)
    yield X.on_each(*qq)
    yield Z(qq[n-1]).controlled_by(*(qq[0:n-1]))
    yield X.on_each(*qq)
    yield H.on_each(*qq)

In [6]:
def grover(trials_number):
    n=3
    qq = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()
    circuit.append(H.on_each(*qq))

    for i in range(2):
        circuit.append(oracle010(qq))
        circuit.append(grover_diffusion(qq,n))
    circuit.append(cirq.measure(*qq, key='result'))

    # determine the statistics of the measurements
    s = cirq.Simulator()
    samples = s.run(circuit, repetitions=trials_number)

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    print(counts)
    return counts.get('010')

In [7]:
# run grover to test if your function gives the right answer
grover(100)

Counter({'010': 95, '001': 2, '000': 1, '110': 1, '100': 1})


95

In [8]:
# hidden tests in this cell will be used for grading.

## Question 2 (6 points)

Graph $G$ has 5 vertices and 6 edges: (0,3), (0,4), (1,3), (1,4), (2,3), (2,4).

Write an oracle for the graph $G$ to check whether it admits a valid 2-coloring.

The function `oracle2` has

* input: `qq`, a 12-qubit register
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits 0-4 for the vertices, 5-10 for the edges and 11 as the ancilla.

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation2` functions.

In [9]:
def oracle2(qq):
    """Oracle for valid 2-coloring: all 6 edges have different colors"""
    # Compute XOR for each edge (flag = 1 if colors differ)
    edges = [(0,3,5), (0,4,6), (1,3,7), (1,4,8), (2,3,9), (2,4,10)]

    for u, v, flag in edges:
        yield cirq.CX(qq[u], qq[flag])
        yield cirq.CX(qq[v], qq[flag])

    # Phase kickback on ancilla if ALL flags are |1>
    yield cirq.H(qq[11])

    # use CCZ on groups or rely on controlled-Z pattern
    controls = [qq[i] for i in range(5,11)]
    yield cirq.Z(qq[11]).controlled_by(*controls)

    yield cirq.H(qq[11])

    # Uncompute flags
    for u, v, flag in reversed(edges):
        yield cirq.CX(qq[v], qq[flag])
        yield cirq.CX(qq[u], qq[flag])

In [10]:
# We need some code so you can check your solution
def oracle_computation2(qq):
    yield oracle2(qq)
    yield Z(qq[11])
    yield inverse(oracle2(qq))

In [11]:
def grover2(trials_number):
    import cirq
    from cirq import X, H, Z, inverse, CX
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(12)
    n=5

    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation2(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [12]:
#You can use this cell to test your solution
shots=1000
grover2(shots)

Counter({'00011': 474,
         '11100': 423,
         '00111': 3,
         '01101': 8,
         '10001': 4,
         '10111': 5,
         '00010': 2,
         '11000': 2,
         '00100': 3,
         '01100': 4,
         '00101': 4,
         '11110': 4,
         '00001': 2,
         '11111': 2,
         '10100': 7,
         '01000': 3,
         '01001': 4,
         '10000': 4,
         '11010': 5,
         '01010': 5,
         '01110': 5,
         '11001': 1,
         '01011': 2,
         '10010': 2,
         '01111': 5,
         '10101': 1,
         '10110': 6,
         '11011': 2,
         '11101': 4,
         '00000': 2,
         '10011': 2})

In [13]:
# hidden tests in this cell will be used for grading.

## Question 3 (10 points)

Graph $G$ has 4 vertices and 5 edges: (0,1), (0,2), (0,3), (1,2), (1,3)

Write an oracle for the graph $G$ to check whether there exists a coloring with at least 4 edges connecting vertices with different colors.

The function `oracle3` has

* input: `qq`, a 13-qubit register
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits
- 0-3 for the vertices,
- 4-8 for the edges,
- 9-11 for the addition (remember we need three qubits here for addition unlike the last question), and
- 12 as the ancilla.

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation3` functions.

In [19]:
def oracle3(qq):
    """Oracle for Max-Cut >= 4"""
    ops = []
    edges = [(0,1,4), (0,2,5), (0,3,6), (1,2,7), (1,3,8)]

    # Step 1: Compute edge flags (1 if different colors)
    for u, v, e in edges:
        ops.append(cirq.CX(qq[u], qq[e]))
        ops.append(cirq.CX(qq[v], qq[e]))

    # Step 2: Count the number of 1s (cut size) into 3-bit counter (qubits 9,10,11)
    for i in range(5):
        flag = qq[4 + i]
        # Increment counter
        ops.append(cirq.CX(flag, qq[9]))
        ops.append(cirq.CCX(flag, qq[9], qq[10]))
        ops.append(cirq.CCX(flag, qq[10], qq[11]))

    # Step 3: Phase kickback if count >= 4 (i.e., qubit 11 is |1|)
    ops.append(cirq.H(qq[12]))
    ops.append(cirq.CX(qq[11], qq[12]))
    ops.append(cirq.H(qq[12]))

    # Step 4: Uncompute counter (reverse operations)
    for i in reversed(range(5)):
        flag = qq[4 + i]
        ops.append(cirq.CCX(flag, qq[10], qq[11]))
        ops.append(cirq.CCX(flag, qq[9], qq[10]))
        ops.append(cirq.CX(flag, qq[9]))

    # Step 5: Uncompute edge flags
    for u, v, e in reversed(edges):
        ops.append(cirq.CX(qq[v], qq[e]))
        ops.append(cirq.CX(qq[u], qq[e]))

    return ops

In [20]:
# We need some code so you can check your solution
def oracle_computation3(qq):
    yield oracle3(qq)
    yield Z(qq[12])
    yield inverse(oracle3(qq))

In [21]:
import cirq
from cirq import X, H, Z, inverse, CX, CCX

def grover3(trials_number):
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(13)
    n=4

    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation3(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [22]:
#You can use this cell to test your solution
shots=1000
grover3(shots)

Counter({'0001': 56,
         '0011': 62,
         '1001': 69,
         '0101': 53,
         '0010': 70,
         '1010': 63,
         '0000': 54,
         '1111': 81,
         '1101': 73,
         '1100': 58,
         '0111': 68,
         '1000': 71,
         '1110': 53,
         '0110': 54,
         '1011': 68,
         '0100': 47})

In [18]:
# hidden tests in this cell will be used for grading.